# Map-Reduce & Parallel Processing in Pure LangChain
### Multi-Agent Architecture: Dynamic Decomposition, Concurrent Map Workers, and Synthesized Reduce

This notebook demonstrates how to implement the **Map-Reduce** pattern with **pure LangChain agents**—achieving the same dynamic fan-out and aggregation capabilities provided by LangGraph's `Send` API.

---

### Architectural Comparison: LangGraph vs. LangChain Map-Reduce

```
┌────────────────────────────────────────────────────────────┐
│                    LANGGRAPH MAP-REDUCE                    │
├────────────────────────────────────────────────────────────┤
│  1. State Reducer : Annotated[list, operator.add]          │
│  2. Fan-out Node  : Returns [Send('worker', task) for ...] │
│  3. Engine        : Pregel executes Send() in 1 superstep  │
│  4. Reduce Node   : Runs automatically at barrier sync     │
└────────────────────────────────────────────────────────────┘

┌────────────────────────────────────────────────────────────┐
│                    LANGCHAIN MAP-REDUCE                    │
├────────────────────────────────────────────────────────────┤
│  1. Orchestrator  : Structured Output Plan (N tasks)       │
│  2. Map Engine    : asyncio.gather() or agent.abatch()     │
│  3. Concurrency   : Parallel async execution with tools    │
│  4. Reducer Agent : Synthesizes collected worker outputs   │
└────────────────────────────────────────────────────────────┘
```

### Use Case: Autonomous Competitive Intelligence & Market Benchmarking
* **Orchestrator**: Dynamically decomposes a market domain into 3–4 key entities with tailored search queries.
* **Map Phase**: Independent **Research Agents** execute concurrently, using real-time Tavily search.
* **Reduce Phase**: A **Synthesizer Agent** aggregates all research into an Executive Summary and Side-by-Side Comparison Matrix.


## 1. Setup & Environment Variables
Load API keys for Google Gemini and Tavily Search.


In [1]:
import os
from getpass import getpass
from dotenv import load_dotenv

# Robustly load .env from current or parent directory
if os.path.exists(".env"):
    load_dotenv(dotenv_path=".env")
elif os.path.exists("../.env"):
    load_dotenv(dotenv_path="../.env")
else:
    load_dotenv()

if "GEMINI_API_KEY" not in os.environ and "GOOGLE_API_KEY" in os.environ:
    os.environ["GEMINI_API_KEY"] = os.environ["GOOGLE_API_KEY"]

print("✓ Environment configured successfully!")
print(f"Google API Key present: {'GEMINI_API_KEY' in os.environ or 'GOOGLE_API_KEY' in os.environ}")
print(f"Tavily API Key present: {'TAVILY_API_KEY' in os.environ}")


✓ Environment configured successfully!
Google API Key present: True
Tavily API Key present: True


## 2. Research Tools
We define a web search tool using Tavily that each parallel worker agent can use.


In [2]:
from langchain.tools import tool
from langchain_community.utilities.tavily_search import TavilySearchAPIWrapper

tavily_wrapper = TavilySearchAPIWrapper()

@tool
def search_web(query: str) -> str:
    """Searches the web for up-to-date technical, architectural, and business information."""
    try:
        results = tavily_wrapper.results(query, max_results=3)
        if not results:
            return f"No results found for query: '{query}'."
        
        output = f"Search Results for '{query}':\n"
        for i, res in enumerate(results, 1):
            output += f"Source {i}: {res.get('title', 'Untitled')}\n"
            output += f"URL: {res.get('url', 'N/A')}\n"
            output += f"Content: {res.get('content', '')}\n\n"
        return output
    except Exception as e:
        return f"Web search encountered an error: {str(e)}"

print("✓ Search tool instantiated.")


✓ Search tool instantiated.


## 3. Pydantic Schemas for Map-Reduce
Here we define the schemas for the Orchestrator's plan and the worker tasks.


In [3]:
from pydantic import BaseModel, Field
from typing import List, Optional

class EntityTask(BaseModel):
    id: int = Field(description="Unique numeric identifier for ordering in the reduce step")
    entity_name: str = Field(description="Name of the competitor, product, or technology to analyze")
    focus_areas: List[str] = Field(description="Key criteria to evaluate: architecture, adoption, features, limitations")
    search_queries: List[str] = Field(description="2-3 specific search queries to gather fresh evidence")

class BenchmarkingPlan(BaseModel):
    market_domain: str = Field(description="The sector or technology category being analyzed")
    evaluation_criteria: List[str] = Field(description="Standardized criteria to evaluate across all competitors")
    entities: List[EntityTask] = Field(description="List of entities to research in parallel")

print("✓ Schemas defined.")


✓ Schemas defined.


## 4. Orchestrator: Dynamic Plan & Task Decomposer
The Orchestrator decomposes any broad market query into a structured `BenchmarkingPlan` with $N$ independent tasks.


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0)

ORCHESTRATOR_PROMPT = """You are a Principal Market Intelligence Architect.
Given a technology domain or comparison topic, produce a structured benchmarking plan:
1. Identify 3 to 4 leading or emerging solutions in this domain.
2. Define 3-5 standardized evaluation criteria (e.g. Architecture, Core Strengths, Ecosystem, Pricing/Licensing).
3. For each entity, specify tailored focus areas and 2 high-signal search queries.

Output MUST strictly adhere to the BenchmarkingPlan schema."""

def create_benchmarking_plan(topic: str) -> BenchmarkingPlan:
    orchestrator = llm.with_structured_output(BenchmarkingPlan)
    plan = orchestrator.invoke([
        SystemMessage(content=ORCHESTRATOR_PROMPT),
        HumanMessage(content=f"Analyze and benchmark the following domain: {topic}")
    ])
    return plan

# Quick test
test_plan = create_benchmarking_plan("AI Agent Frameworks")
print(f"Domain: {test_plan.market_domain}")
print(f"Criteria: {', '.join(test_plan.evaluation_criteria)}")
print("Planned Entities to map:")
for e in test_plan.entities:
    print(f"  [{e.id}] {e.entity_name} (Queries: {e.search_queries})")


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Domain: AI Agent Frameworks
Criteria: Architecture & Modularity, Core Capabilities & Use Cases, Ecosystem & Integrations, Developer Experience & Community Support, Scalability & Production Readiness
Planned Entities to map:
  [1] LangChain (Queries: ['LangChain agent architecture overview', 'LangChain production deployment best practices'])
  [2] LlamaIndex (Queries: ['LlamaIndex RAG pipeline architecture', 'LlamaIndex data agent use cases'])
  [3] AutoGen (Queries: ['AutoGen multi-agent collaboration examples', 'AutoGen human feedback integration'])
  [4] Haystack (Queries: ['Haystack RAG pipeline architecture', 'Haystack enterprise features comparison'])


## 5. Worker Agent: The "Map" Entity Researcher
Each worker is a full LangChain agent with access to web search tools. It focuses exclusively on researching one entity according to the assigned criteria.


In [5]:
from langchain.agents import create_agent

WORKER_SYSTEM_PROMPT = """You are an Expert Technical Research Analyst.
Your task is to conduct an in-depth, factual investigation of a single company, tool, or framework.

Instructions:
1. Use the search_web tool with the provided queries to gather authoritative information.
2. Analyze the entity against the provided evaluation criteria.
3. Structure your response in clean Markdown with:
   - **Executive Overview**: What is it and its primary design philosophy?
   - **Key Capabilities & Architecture**: Technical strengths.
   - **Limitations / Trade-offs**: What are its weaknesses or common complaints?
   - **Target Audience / Best Fit**: When should an engineering team choose it?
4. Keep the report factual, technical, and concise (200-300 words). Cite sources at the end."""

# Instantiate worker agent
worker_agent = create_agent(
    model=llm,
    tools=[search_web],
    system_prompt=WORKER_SYSTEM_PROMPT
)

print("✓ Map Worker Agent instantiated.")


✓ Map Worker Agent instantiated.


## 6. The Parallel Map Phase (`asyncio.gather` / `.abatch`)
This is the core parallel processing engine. It takes the list of $N$ entities generated by the Orchestrator and dispatches research agents concurrently.


In [6]:
import asyncio
import time
from rich.console import Console
from rich.panel import Panel

console = Console()

async def research_single_entity_async(task: EntityTask, criteria: List[str]) -> dict:
    """Worker task executor: runs the agent for a single entity."""
    start = time.time()
    console.print(f"[bold cyan]>>> STARTING MAP WORKER for: {task.entity_name}[/bold cyan]")
    
    query = (
        f"Entity to research: {task.entity_name}\n"
        f"Evaluation Criteria: {', '.join(criteria)}\n"
        f"Specific Focus Areas: {', '.join(task.focus_areas)}\n"
        f"Suggested Search Queries: {task.search_queries}\n"
    )
    
    # Invoke the agent asynchronously
    response = await worker_agent.ainvoke({"messages": [{"role": "user", "content": query}]})
    
    # Extract final text
    last_msg = response["messages"][-1]
    content = last_msg.content[0]["text"] if isinstance(last_msg.content, list) else last_msg.content
    
    elapsed = time.time() - start
    console.print(f"[bold green]<<< COMPLETED MAP WORKER for: {task.entity_name} in {elapsed:.2f}s[/bold green]")
    
    return {
        "id": task.id,
        "entity_name": task.entity_name,
        "content": content,
        "elapsed_seconds": elapsed
    }

async def execute_parallel_map(plan: BenchmarkingPlan) -> List[dict]:
    """Dispatches all entity research tasks in parallel, waiting for all to complete."""
    start_total = time.time()
    console.print("\n" + "="*80, style="bold magenta")
    console.print(f"🚀 EXECUTING PARALLEL MAP PHASE: {len(plan.entities)} Concurrent Workers", style="bold yellow")
    console.print("="*80, style="bold magenta")
    
    # Run all workers concurrently using asyncio.gather
    tasks = [
        research_single_entity_async(entity, plan.evaluation_criteria)
        for entity in plan.entities
    ]
    results = await asyncio.gather(*tasks)
    
    total_time = time.time() - start_total
    
    # Sort results by task ID (deterministic ordering, exactly like LangGraph state reducer)
    results = sorted(results, key=lambda x: x["id"])
    
    sum_individual_time = sum(r["elapsed_seconds"] for r in results)
    console.print("\n" + "─"*80, style="bold green")
    console.print(f"✨ ALL {len(results)} WORKERS FINISHED!", style="bold green")
    console.print(f"⏱️  Total Parallel Execution Time : {total_time:.2f} seconds")
    console.print(f"⏱️  Sum of Individual Task Times   : {sum_individual_time:.2f} seconds")
    console.print(f"⚡ Speedup Ratio                   : {sum_individual_time / total_time:.2f}x faster than sequential!", style="bold yellow")
    console.print("─"*80, style="bold green")
    
    return results

print("✓ Parallel Map Execution Engine defined.")


✓ Parallel Map Execution Engine defined.


## 7. Reducer Agent: Synthesizing Parallel Findings
The Reducer receives all individual entity research reports, sorts them, and compiles a comprehensive Executive Market Benchmark with a side-by-side comparison matrix.


In [7]:
REDUCER_SYSTEM_PROMPT = """You are the Principal Synthesis & Market Strategy Officer.
Your job is to read individual research profiles of competing entities and produce a comprehensive, publication-grade Comparative Benchmark Report.

Hard Requirements:
1. Title: Clean H1 (# [Market Domain] Comparative Benchmark Report).
2. Executive Summary: 100-150 words synthesizing the state of the market and major architectural trends.
3. Side-by-Side Comparison Matrix: A detailed Markdown Table comparing all analyzed entities across standard dimensions (Architecture, Core Strengths, Limitations, Best Use Case).
4. Deep-Dive Entity Profiles: Clean H2 sections for each entity, preserving key technical facts from the workers.
5. Strategic Recommendation & Verdict: Clear decision framework for teams choosing between these options.

Ensure all special characters like '$' are escaped as '\\$' for clean rendering."""

reducer_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=REDUCER_SYSTEM_PROMPT
)

async def reduce_and_compile_report(plan: BenchmarkingPlan, mapped_results: List[dict]) -> str:
    """Compiles mapped worker outputs into the final synthesized benchmark."""
    console.print("\n[bold yellow]🔄 EXECUTING REDUCE PHASE (Synthesizer Agent)...[/bold yellow]")
    
    combined_research = f"Market Domain: {plan.market_domain}\n"
    combined_research += f"Evaluation Criteria: {', '.join(plan.evaluation_criteria)}\n\n"
    
    for r in mapped_results:
        combined_research += f"### Entity: {r['entity_name']}\n"
        combined_research += f"{r['content']}\n\n"
        combined_research += "---\n"
        
    response = await reducer_agent.ainvoke({
        "messages": [{"role": "user", "content": combined_research}]
    })
    
    last_msg = response["messages"][-1]
    final_report = last_msg.content[0]["text"] if isinstance(last_msg.content, list) else last_msg.content
    console.print("[bold green]✓ REDUCE PHASE COMPLETE![/bold green]")
    return final_report

print("✓ Reducer Agent defined.")


✓ Reducer Agent defined.


## 8. Complete End-to-End Execution Pipeline
We tie Orchestration, Parallel Mapping, and Reduction together and save the final report to disk.


In [ ]:
import re

def save_report_to_disk(title: str, content: str) -> str:
    os.makedirs("reports", exist_ok=True)
    safe_title = re.sub(r'[^a-zA-Z0-9\s\_\-]', '', title).strip().replace(" ", "_")
    filepath = f"reports/{safe_title}.md"
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)
    return filepath

async def run_map_reduce_agent(topic: str):
    console.print("\n" + "#"*80, style="bold blue")
    console.print(f"🎯 STARTING MAP-REDUCE MULTI-AGENT PIPELINE FOR: '{topic}'", style="bold blue")
    console.print("#"*80, style="bold blue")
    
    # 1. Orchestrator Plan
    console.print("\n[bold cyan]1. Orchestrator formulating plan...[/bold cyan]")
    plan = create_benchmarking_plan(topic)
    console.print(f"Planned {len(plan.entities)} entities: {[e.entity_name for e in plan.entities]}")
    
    # 2. Parallel Map
    mapped_results = await execute_parallel_map(plan)
    
    # 3. Reduce
    final_report = await reduce_and_compile_report(plan, mapped_results)
    
    # 4. Save
    saved_path = save_report_to_disk(f"{plan.market_domain}_Benchmark", final_report)
    console.print(f"\n💾 [bold green]Saved Benchmark Report to: {saved_path}[/bold green]")
    
    return final_report, saved_path

# Execute the full pipeline on a cutting-edge domain
test_domain = "AI Agent Frameworks: LangGraph, AutoGen, and CrewAI"
final_report, path = await run_map_reduce_agent(test_domain)

print("\n" + "="*80)
print("EXECUTIVE BENCHMARK DELIVERABLE (PREVIEW):\n")
print(final_report[:1200] + "\n\n[... Full Report Saved on Disk ...]")
